In [2]:

# =====================================================
# Week 2 - Phase 1 - Cell 1: Model Availability Check
# =====================================================

from sklearn.pipeline import Pipeline

# Step 1: Check model availability
model_exists = "final_model" in globals()
threshold_exists = "final_threshold" in globals()

print("Week 1 Model Available:", model_exists)
print("Threshold Available:", threshold_exists)

# Step 2: Verify trained model
if model_exists and threshold_exists:

    print("\nModel Type:", type(
        final_model.named_steps["classifier"]
    ).__name__)

    print("Selected Threshold:", final_threshold)

    print(
        "Total Input Features:",
        len(final_model.feature_names_in_)
    )

    print("\nMODEL READY FOR WEEK 2!")

else:
    print("\nMODEL NOT LOADED!")
    print("Open the Week 1 notebook.")
    print("Run the training cells through Phase 4 Cell 10.")


Week 1 Model Available: False
Threshold Available: False

MODEL NOT LOADED!
Open the Week 1 notebook.
Run the training cells through Phase 4 Cell 10.


In [3]:

# =====================================================
# Week 2 - Phase 1 - Cell 2: Check Existing Files
# =====================================================

from pathlib import Path

project_dir = Path(r"D:\Ai_Ml_intren")

print("PROJECT FILE VERIFICATION")
print("=" * 50)

for folder_name in ["week_1", "week_2"]:

    folder = project_dir / folder_name

    print(f"\nFolder: {folder}")

    if not folder.exists():
        print("Folder Not Found!")
        continue

    for item in sorted(folder.iterdir()):

        if item.suffix.lower() in [
            ".ipynb", ".csv", ".joblib", ".pkl"
        ] or item.name == "outputs":

            file_type = "FILE" if item.is_file() else "FOLDER"
            print(f"[{file_type}] {item.name}")

print("\n" + "=" * 50)
print("File Verification Completed!")


PROJECT FILE VERIFICATION

Folder: D:\Ai_Ml_intren\week_1
[FOLDER] notebook.ipynb
[FOLDER] outputs
[FILE] Telco-Customer-Churn.csv

Folder: D:\Ai_Ml_intren\week_2
[FILE] notebook.ipynb

File Verification Completed!


In [4]:

# =====================================================
# Week 2 - Phase 1 - Cell 3
# Rebuild and Train the Week 1 Random Forest Model
# =====================================================

import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Step 1: Load Week 1 dataset
dataset_path = Path(
    r"D:\Ai_Ml_intren\week_1\Telco-Customer-Churn.csv"
)

df = pd.read_csv(dataset_path)

print("Original Dataset Shape:", df.shape)

# Step 2: Handle missing TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Week 1 business rule: missing bills for tenure=0
missing_mask = df["TotalCharges"].isna()

assert (df.loc[missing_mask, "tenure"] == 0).all()

df.loc[missing_mask, "TotalCharges"] = 0.0

# Step 3: Encode target variable
df["ChurnEncoded"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Step 4: Recreate Week 1 engineered features
df["EstimatedLifetimeBilling"] = (
    df["MonthlyCharges"] * df["tenure"]
)

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, float("inf")],
    labels=[
        "New Customers (0-12)",
        "Short-Term (13-24)",
        "Mid-Term (25-48)",
        "Long-Term (49+)"
    ]
)

service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["NumAddOnServices"] = (
    df[service_columns].eq("Yes").sum(axis=1)
)

# Step 5: Separate features and target
X = df.drop(
    columns=["customerID", "Churn", "ChurnEncoded"]
)

y = df["ChurnEncoded"]

# Step 6: Recreate stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Step 7: Identify feature types
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Step 8: Build preprocessing pipelines
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Step 9: Recreate tuned Random Forest pipeline
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        max_depth=4,
        min_samples_split=5,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=1
    ))
])

# Step 10: Train using training data only
final_model.fit(X_train, y_train)

# Step 11: Restore the Week 1 selected threshold
final_threshold = 0.36

# Step 12: Verify trained model
assert X.shape == (7043, 22)
assert X_train.shape == (5634, 22)
assert hasattr(
    final_model.named_steps["classifier"],
    "estimators_"
)

print("\n" + "=" * 50)
print("WEEK 2 MODEL TRAINING RESULTS")
print("=" * 50)

print("Total Customers:", len(df))
print("Input Features:", X.shape[1])
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("\nModel: RandomForestClassifier")
print("Number of Trees:", 150)
print("Maximum Depth:", 4)
print("Selected Threshold:", final_threshold)

print("\nMODEL TRAINED SUCCESSFULLY!")
print("Ready for Serialization.")


Original Dataset Shape: (7043, 21)

WEEK 2 MODEL TRAINING RESULTS
Total Customers: 7043
Input Features: 22
Training Samples: 5634
Testing Samples: 1409

Model: RandomForestClassifier
Number of Trees: 150
Maximum Depth: 4
Selected Threshold: 0.36

MODEL TRAINED SUCCESSFULLY!
Ready for Serialization.


In [5]:

# =====================================================
# Week 2 - Phase 2 - Cell 1
# Model Serialization using Pickle and Joblib
# =====================================================

import pickle
import joblib
import numpy as np
import pandas as pd
import sklearn

from pathlib import Path

# Step 1: Define output directory
output_dir = Path(r"D:\Ai_Ml_intren\week_2")
output_dir.mkdir(parents=True, exist_ok=True)

# Step 2: Prepare model package
# Save the model, threshold, and expected features together
model_package = {
    "model": final_model,
    "threshold": float(final_threshold),
    "feature_names": X_train.columns.tolist(),
    "sklearn_version": sklearn.__version__,
    "pandas_version": pd.__version__
}

# Step 3: Define output file paths
pickle_path = output_dir / "model_v1.pkl"
joblib_path = output_dir / "model_v1.joblib"

# Step 4: Serialize using Pickle
with open(pickle_path, "wb") as file:
    pickle.dump(
        model_package,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Pickle Model Saved Successfully!")

# Step 5: Serialize using Joblib
joblib.dump(
    model_package,
    joblib_path,
    compress=0
)

print("Joblib Model Saved Successfully!")

# Step 6: Compare file sizes
pickle_size = pickle_path.stat().st_size / 1024
joblib_size = joblib_path.stat().st_size / 1024

print("\nMODEL FILE SIZE COMPARISON")
print("=" * 45)

print(f"Pickle File Size : {pickle_size:.2f} KB")
print(f"Joblib File Size : {joblib_size:.2f} KB")

# Step 7: Reload both saved packages
# Only load files created by this trusted script
with open(pickle_path, "rb") as file:
    loaded_pickle = pickle.load(file)

loaded_joblib = joblib.load(joblib_path)

# Step 8: Verify both loaded models
sample_data = X_train.head(5)

original_probs = final_model.predict_proba(
    sample_data
)

pickle_probs = loaded_pickle["model"].predict_proba(
    sample_data
)

joblib_probs = loaded_joblib["model"].predict_proba(
    sample_data
)

assert np.allclose(original_probs, pickle_probs)
assert np.allclose(original_probs, joblib_probs)

assert loaded_joblib["threshold"] == final_threshold
assert len(loaded_joblib["feature_names"]) == 22

# Step 9: Display final verification
print("\nMODEL SERIALIZATION VERIFICATION")
print("=" * 45)

print("Pickle File Exists :", pickle_path.is_file())
print("Joblib File Exists :", joblib_path.is_file())

print("Original vs Pickle Predictions : MATCH")
print("Original vs Joblib Predictions : MATCH")

print("\nSaved Files:")
print(pickle_path)
print(joblib_path)

print("\nMODEL SERIALIZATION COMPLETED SUCCESSFULLY!")


Pickle Model Saved Successfully!
Joblib Model Saved Successfully!

MODEL FILE SIZE COMPARISON
Pickle File Size : 418.67 KB
Joblib File Size : 438.92 KB

MODEL SERIALIZATION VERIFICATION
Pickle File Exists : True
Joblib File Exists : True
Original vs Pickle Predictions : MATCH
Original vs Joblib Predictions : MATCH

Saved Files:
D:\Ai_Ml_intren\week_2\model_v1.pkl
D:\Ai_Ml_intren\week_2\model_v1.joblib

MODEL SERIALIZATION COMPLETED SUCCESSFULLY!
